In [0]:
from pyspark.sql import functions as F
from datetime import datetime, timezone
import uuid

BRONZE_SCHEMA = "supplysage_bronze"

SYNTHETIC_BATCH_ID = f"synth_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}_{str(uuid.uuid4())[:8]}"
GENERATION_VERSION = "v1"
RANDOM_SEED = 42
CREATED_BY = "Vigya"
CREATED_AT = datetime.now(timezone.utc).isoformat()
NOTEBOOK_NAME = "04_bronze_lineage_and_relationship_validation"

print("SYNTHETIC_BATCH_ID:", SYNTHETIC_BATCH_ID)
print("CREATED_AT:", CREATED_AT)

In [0]:
SYNTHETIC_TABLES = [
    {
        "table_name": "bronze_suppliers",
        "generation_rule": "category_and_trade_lane_weighted_supplier_master_generation"
    },
    {
        "table_name": "bronze_supplier_aliases",
        "generation_rule": "supplier_alias_generation_for_entity_resolution"
    },
    {
        "table_name": "bronze_supplier_sku_map",
        "generation_rule": "m5_sku_weighted_supplier_assignment_with_primary_and_alternate_mapping"
    },
    {
        "table_name": "bronze_alternate_suppliers",
        "generation_rule": "alternate_supplier_qualification_generation"
    },
    {
        "table_name": "bronze_supplier_scorecards",
        "generation_rule": "monthly_supplier_performance_generation"
    },
    {
        "table_name": "bronze_purchase_orders",
        "generation_rule": "purchase_order_generation_from_supplier_sku_and_lead_time_logic"
    },
    {
        "table_name": "bronze_shipment_routes",
        "generation_rule": "supplier_route_generation_from_origin_country_transport_mode_and_risk_region"
    }
]

SOURCE_TABLES_USED = [
    "bronze_m5_sales_train_validation",
    "bronze_m5_calendar",
    "bronze_m5_sell_prices",
    "bronze_retail_inventory",
    "bronze_dataco_supply_chain",
    "bronze_dq_table_profiles",
    "bronze_source_contracts"
]


def full_table(table_name):
    return f"{BRONZE_SCHEMA}.{table_name}"


def table_exists(table_name):
    return spark.catalog.tableExists(full_table(table_name))

In [0]:
inspection_rows = []

for t in SYNTHETIC_TABLES:
    table_name = t["table_name"]

    if table_exists(table_name):
        df = spark.table(full_table(table_name))
        inspection_rows.append({
            "table_name": table_name,
            "exists": True,
            "row_count": df.count(),
            "column_count": len(df.columns),
            "columns": ", ".join(df.columns)
        })
    else:
        inspection_rows.append({
            "table_name": table_name,
            "exists": False,
            "row_count": None,
            "column_count": None,
            "columns": None
        })

inspection_df = spark.createDataFrame(inspection_rows)
display(inspection_df)

In [0]:
for t in SYNTHETIC_TABLES:
    table_name = t["table_name"]

    if not table_exists(table_name):
        print(f"Skipping {table_name}: table does not exist.")
        continue

    df = spark.table(full_table(table_name))
    existing_cols = df.columns

    if "synthetic_batch_id" not in existing_cols:
        spark.sql(f"""
            ALTER TABLE {full_table(table_name)}
            ADD COLUMNS (synthetic_batch_id STRING)
        """)
        print(f"Added synthetic_batch_id to {table_name}")

    if "is_synthetic" not in existing_cols:
        spark.sql(f"""
            ALTER TABLE {full_table(table_name)}
            ADD COLUMNS (is_synthetic BOOLEAN)
        """)
        print(f"Added is_synthetic to {table_name}")

    spark.sql(f"""
        UPDATE {full_table(table_name)}
        SET
            synthetic_batch_id = COALESCE(synthetic_batch_id, '{SYNTHETIC_BATCH_ID}'),
            is_synthetic = COALESCE(is_synthetic, true)
    """)

    print(f"Updated lineage fields for {table_name}")

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {BRONZE_SCHEMA}.bronze_product_crosswalk AS

SELECT DISTINCT
    CONCAT('M5_', item_id) AS canonical_sku_id,
    'm5' AS source_system,
    CAST(item_id AS STRING) AS source_product_id,
    CAST(NULL AS STRING) AS source_product_name,
    CAST(cat_id AS STRING) AS source_category,
    CAST(dept_id AS STRING) AS source_department,
    1.00 AS confidence,
    'native_m5_item_mapping' AS mapping_rule,
    '{SYNTHETIC_BATCH_ID}' AS synthetic_batch_id,
    true AS is_synthetic
FROM {BRONZE_SCHEMA}.bronze_m5_sales_train_validation

UNION ALL

SELECT DISTINCT
    CONCAT('RET_', CAST(`Product ID` AS STRING)) AS canonical_sku_id,
    'retail_inventory' AS source_system,
    CAST(`Product ID` AS STRING) AS source_product_id,
    CAST(NULL AS STRING) AS source_product_name,
    CAST(Category AS STRING) AS source_category,
    CAST(NULL AS STRING) AS source_department,
    1.00 AS confidence,
    'native_retail_inventory_product_mapping' AS mapping_rule,
    '{SYNTHETIC_BATCH_ID}' AS synthetic_batch_id,
    true AS is_synthetic
FROM {BRONZE_SCHEMA}.bronze_retail_inventory

UNION ALL

SELECT DISTINCT
    CONCAT('DATACO_', CAST(`Product Card Id` AS STRING)) AS canonical_sku_id,
    'dataco' AS source_system,
    CAST(`Product Card Id` AS STRING) AS source_product_id,
    CAST(`Product Name` AS STRING) AS source_product_name,
    CAST(`Category Name` AS STRING) AS source_category,
    CAST(`Department Name` AS STRING) AS source_department,
    1.00 AS confidence,
    'native_dataco_product_mapping' AS mapping_rule,
    '{SYNTHETIC_BATCH_ID}' AS synthetic_batch_id,
    true AS is_synthetic
FROM {BRONZE_SCHEMA}.bronze_dataco_supply_chain
WHERE `Product Card Id` IS NOT NULL
""")

display(spark.table(f"{BRONZE_SCHEMA}.bronze_product_crosswalk").limit(20))

In [0]:
from pyspark.sql import functions as F
from datetime import datetime, timezone
import uuid

BRONZE_SCHEMA = "supplysage_bronze"

SYNTHETIC_BATCH_ID = f"synth_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}_{str(uuid.uuid4())[:8]}"
GENERATION_VERSION = "v1"
RANDOM_SEED = 42
CREATED_BY = "Vigya"
CREATED_AT = datetime.now(timezone.utc).isoformat()
NOTEBOOK_NAME = "04_bronze_lineage_and_relationship_validation"

SYNTHETIC_TABLES = [
    {
        "table_name": "bronze_suppliers",
        "generation_rule": "category_and_trade_lane_weighted_supplier_master_generation"
    },
    {
        "table_name": "bronze_supplier_aliases",
        "generation_rule": "supplier_alias_generation_for_entity_resolution"
    },
    {
        "table_name": "bronze_supplier_sku_map",
        "generation_rule": "m5_sku_weighted_supplier_assignment_with_primary_and_alternate_mapping"
    },
    {
        "table_name": "bronze_alternate_suppliers",
        "generation_rule": "alternate_supplier_qualification_generation"
    },
    {
        "table_name": "bronze_supplier_scorecards",
        "generation_rule": "monthly_supplier_performance_generation"
    },
    {
        "table_name": "bronze_purchase_orders",
        "generation_rule": "purchase_order_generation_from_supplier_sku_and_lead_time_logic"
    },
    {
        "table_name": "bronze_shipment_routes",
        "generation_rule": "supplier_route_generation_from_origin_country_transport_mode_and_risk_region"
    }
]

SOURCE_TABLES_USED = [
    "bronze_m5_sales_train_validation",
    "bronze_m5_calendar",
    "bronze_m5_sell_prices",
    "bronze_retail_inventory",
    "bronze_dataco_supply_chain",
    "bronze_dq_table_profiles",
    "bronze_source_contracts"
]

def full_table(table_name):
    return f"{BRONZE_SCHEMA}.{table_name}"

def table_exists(table_name):
    return spark.catalog.tableExists(full_table(table_name))

print("SYNTHETIC_BATCH_ID:", SYNTHETIC_BATCH_ID)
print("CREATED_AT:", CREATED_AT)

In [0]:
manifest_rows = []

for t in SYNTHETIC_TABLES:
    table_name = t["table_name"]

    if table_exists(table_name):
        row_count = spark.table(full_table(table_name)).count()
        status = "CREATED"
    else:
        row_count = None
        status = "MISSING"

    manifest_rows.append({
        "synthetic_batch_id": SYNTHETIC_BATCH_ID,
        "target_table": table_name,
        "generation_version": GENERATION_VERSION,
        "generation_rule": t["generation_rule"],
        "source_tables_used": ", ".join(SOURCE_TABLES_USED),
        "random_seed": RANDOM_SEED,
        "row_count": row_count,
        "status": status,
        "created_at": CREATED_AT,
        "created_by": CREATED_BY,
        "notebook_name": NOTEBOOK_NAME
    })

manifest_df = spark.createDataFrame(manifest_rows)

(
    manifest_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{BRONZE_SCHEMA}.bronze_synthetic_generation_manifest")
)

display(manifest_df.orderBy("target_table"))

In [0]:
%sql
SELECT *
FROM supplysage_bronze.bronze_synthetic_generation_manifest
ORDER BY target_table;

In [0]:
crosswalk_count = spark.table(f"{BRONZE_SCHEMA}.bronze_product_crosswalk").count()

crosswalk_manifest_df = spark.createDataFrame([{
    "synthetic_batch_id": SYNTHETIC_BATCH_ID,
    "target_table": "bronze_product_crosswalk",
    "generation_version": GENERATION_VERSION,
    "generation_rule": "canonical_product_crosswalk_generation_from_m5_retail_inventory_and_dataco",
    "source_tables_used": ", ".join([
        "bronze_m5_sales_train_validation",
        "bronze_retail_inventory",
        "bronze_dataco_supply_chain"
    ]),
    "random_seed": RANDOM_SEED,
    "row_count": crosswalk_count,
    "status": "CREATED",
    "created_at": CREATED_AT,
    "created_by": CREATED_BY,
    "notebook_name": NOTEBOOK_NAME
}])

if table_exists("bronze_synthetic_generation_manifest"):
    existing_manifest_df = spark.table(f"{BRONZE_SCHEMA}.bronze_synthetic_generation_manifest")

    # Remove old crosswalk row if this cell is rerun.
    existing_manifest_df = existing_manifest_df.filter(
        F.col("target_table") != "bronze_product_crosswalk"
    )

    updated_manifest_df = existing_manifest_df.unionByName(crosswalk_manifest_df)
else:
    updated_manifest_df = crosswalk_manifest_df

(
    updated_manifest_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{BRONZE_SCHEMA}.bronze_synthetic_generation_manifest")
)

display(
    spark.table(f"{BRONZE_SCHEMA}.bronze_synthetic_generation_manifest")
    .orderBy("target_table")
)

In [0]:
from pyspark.sql import functions as F

BRONZE_SCHEMA = "supplysage_bronze"

def full_table(table_name):
    return f"{BRONZE_SCHEMA}.{table_name}"

def table_exists(table_name):
    return spark.catalog.tableExists(full_table(table_name))

def column_exists(table_name, column_name):
    if not table_exists(table_name):
        return False
    return column_name in spark.table(full_table(table_name)).columns

def scalar(sql_text):
    return spark.sql(sql_text).collect()[0][0]

validation_rows = []

def add_check(check_name, status, metric_value, notes):
    validation_rows.append({
        "synthetic_batch_id": SYNTHETIC_BATCH_ID,
        "validated_at": CREATED_AT,
        "check_name": check_name,
        "status": status,
        "metric_value": str(metric_value),
        "notes": notes
    })

SYNTHETIC_TABLES = [
    "bronze_suppliers",
    "bronze_supplier_aliases",
    "bronze_supplier_sku_map",
    "bronze_alternate_suppliers",
    "bronze_supplier_scorecards",
    "bronze_purchase_orders",
    "bronze_shipment_routes",
    "bronze_product_crosswalk",
    "bronze_synthetic_generation_manifest"
]

In [0]:
for table_name in SYNTHETIC_TABLES:
    if not table_exists(table_name):
        add_check(
            f"{table_name}_exists",
            "FAIL",
            0,
            f"{table_name} does not exist."
        )
    else:
        row_count = spark.table(full_table(table_name)).count()
        add_check(
            f"{table_name}_row_count_positive",
            "PASS" if row_count > 0 else "FAIL",
            row_count,
            f"{table_name} should have at least one row."
        )

display(spark.createDataFrame(validation_rows))

In [0]:
supplier_duplicate_count = scalar(f"""
SELECT COUNT(*)
FROM (
    SELECT supplier_id, COUNT(*) AS cnt
    FROM {BRONZE_SCHEMA}.bronze_suppliers
    GROUP BY supplier_id
    HAVING cnt > 1
)
""")

add_check(
    "supplier_id_unique",
    "PASS" if supplier_duplicate_count == 0 else "FAIL",
    supplier_duplicate_count,
    "bronze_suppliers should have one row per supplier_id."
)

supplier_null_count = scalar(f"""
SELECT COUNT(*)
FROM {BRONZE_SCHEMA}.bronze_suppliers
WHERE supplier_id IS NULL
""")

add_check(
    "supplier_id_not_null",
    "PASS" if supplier_null_count == 0 else "FAIL",
    supplier_null_count,
    "supplier_id should never be null."
)

display(spark.createDataFrame(validation_rows).filter(F.col("check_name").contains("supplier_id")))

In [0]:
bad_alias_supplier_count = scalar(f"""
SELECT COUNT(*)
FROM {BRONZE_SCHEMA}.bronze_supplier_aliases a
LEFT JOIN {BRONZE_SCHEMA}.bronze_suppliers s
    ON a.supplier_id = s.supplier_id
WHERE s.supplier_id IS NULL
""")

add_check(
    "supplier_aliases_reference_valid_suppliers",
    "PASS" if bad_alias_supplier_count == 0 else "FAIL",
    bad_alias_supplier_count,
    "Every supplier alias should reference a valid supplier."
)

if column_exists("bronze_supplier_aliases", "match_confidence"):
    bad_alias_confidence_count = scalar(f"""
    SELECT COUNT(*)
    FROM {BRONZE_SCHEMA}.bronze_supplier_aliases
    WHERE match_confidence < 0 OR match_confidence > 1
    """)

    add_check(
        "supplier_alias_match_confidence_between_zero_and_one",
        "PASS" if bad_alias_confidence_count == 0 else "FAIL",
        bad_alias_confidence_count,
        "match_confidence should be between 0 and 1."
    )

display(spark.createDataFrame(validation_rows).filter(F.col("check_name").contains("alias")))

In [0]:
bad_map_supplier_count = scalar(f"""
SELECT COUNT(*)
FROM {BRONZE_SCHEMA}.bronze_supplier_sku_map m
LEFT JOIN {BRONZE_SCHEMA}.bronze_suppliers s
    ON m.supplier_id = s.supplier_id
WHERE s.supplier_id IS NULL
""")

add_check(
    "supplier_sku_map_references_valid_suppliers",
    "PASS" if bad_map_supplier_count == 0 else "FAIL",
    bad_map_supplier_count,
    "Every supplier-SKU relationship should reference a valid supplier."
)

bad_map_sku_count = scalar(f"""
SELECT COUNT(*)
FROM {BRONZE_SCHEMA}.bronze_supplier_sku_map m
LEFT JOIN (
    SELECT DISTINCT item_id
    FROM {BRONZE_SCHEMA}.bronze_m5_sales_train_validation
) sales
    ON m.sku_id = sales.item_id
WHERE sales.item_id IS NULL
""")

add_check(
    "supplier_sku_map_references_valid_m5_skus",
    "PASS" if bad_map_sku_count == 0 else "FAIL",
    bad_map_sku_count,
    "Every supplier_sku_map.sku_id should exist as an M5 item_id."
)

bad_primary_sku_count = scalar(f"""
SELECT COUNT(*)
FROM (
    SELECT
        sku_id,
        SUM(CASE WHEN is_primary_supplier = true THEN 1 ELSE 0 END) AS primary_count
    FROM {BRONZE_SCHEMA}.bronze_supplier_sku_map
    GROUP BY sku_id
    HAVING primary_count != 1
)
""")

add_check(
    "each_mapped_sku_has_exactly_one_primary_supplier",
    "PASS" if bad_primary_sku_count == 0 else "FAIL",
    bad_primary_sku_count,
    "Each mapped SKU should have exactly one primary supplier."
)

bad_dependency_count = scalar(f"""
SELECT COUNT(*)
FROM {BRONZE_SCHEMA}.bronze_supplier_sku_map
WHERE dependency_percent < 0 OR dependency_percent > 1
""")

add_check(
    "dependency_percent_between_zero_and_one",
    "PASS" if bad_dependency_count == 0 else "FAIL",
    bad_dependency_count,
    "dependency_percent should be between 0 and 1."
)

dependency_sum_issue_count = scalar(f"""
SELECT COUNT(*)
FROM (
    SELECT
        sku_id,
        SUM(dependency_percent) AS dependency_sum
    FROM {BRONZE_SCHEMA}.bronze_supplier_sku_map
    GROUP BY sku_id
    HAVING dependency_sum < 0.95 OR dependency_sum > 1.05
)
""")

add_check(
    "dependency_percent_sums_to_approximately_one_per_sku",
    "PASS" if dependency_sum_issue_count == 0 else "WARN",
    dependency_sum_issue_count,
    "Dependency percentages should sum to approximately 1.0 per SKU. WARN is acceptable if generation logic intentionally models partial coverage."
)

display(spark.createDataFrame(validation_rows).filter(F.col("check_name").contains("sku")))

In [0]:
bad_alt_primary_count = scalar(f"""
SELECT COUNT(*)
FROM {BRONZE_SCHEMA}.bronze_alternate_suppliers a
LEFT JOIN {BRONZE_SCHEMA}.bronze_suppliers s
    ON a.primary_supplier_id = s.supplier_id
WHERE s.supplier_id IS NULL
""")

add_check(
    "alternate_suppliers_reference_valid_primary_suppliers",
    "PASS" if bad_alt_primary_count == 0 else "FAIL",
    bad_alt_primary_count,
    "Every primary_supplier_id should exist in bronze_suppliers."
)

bad_alt_supplier_count = scalar(f"""
SELECT COUNT(*)
FROM {BRONZE_SCHEMA}.bronze_alternate_suppliers a
LEFT JOIN {BRONZE_SCHEMA}.bronze_suppliers s
    ON a.alternate_supplier_id = s.supplier_id
WHERE s.supplier_id IS NULL
""")

add_check(
    "alternate_suppliers_reference_valid_alternate_suppliers",
    "PASS" if bad_alt_supplier_count == 0 else "FAIL",
    bad_alt_supplier_count,
    "Every alternate_supplier_id should exist in bronze_suppliers."
)

bad_alt_self_count = scalar(f"""
SELECT COUNT(*)
FROM {BRONZE_SCHEMA}.bronze_alternate_suppliers
WHERE primary_supplier_id = alternate_supplier_id
""")

add_check(
    "alternate_supplier_not_same_as_primary",
    "PASS" if bad_alt_self_count == 0 else "FAIL",
    bad_alt_self_count,
    "Alternate supplier cannot be the same as primary supplier."
)

bad_alt_sku_count = scalar(f"""
SELECT COUNT(*)
FROM {BRONZE_SCHEMA}.bronze_alternate_suppliers a
LEFT JOIN (
    SELECT DISTINCT item_id
    FROM {BRONZE_SCHEMA}.bronze_m5_sales_train_validation
) sales
    ON a.sku_id = sales.item_id
WHERE sales.item_id IS NULL
""")

add_check(
    "alternate_suppliers_reference_valid_m5_skus",
    "PASS" if bad_alt_sku_count == 0 else "FAIL",
    bad_alt_sku_count,
    "Every alternate supplier SKU should exist as an M5 item_id."
)

if column_exists("bronze_alternate_suppliers", "capacity_available_pct"):
    bad_capacity_count = scalar(f"""
    SELECT COUNT(*)
    FROM {BRONZE_SCHEMA}.bronze_alternate_suppliers
    WHERE capacity_available_pct < 0 OR capacity_available_pct > 1
    """)

    add_check(
        "alternate_capacity_available_pct_between_zero_and_one",
        "PASS" if bad_capacity_count == 0 else "FAIL",
        bad_capacity_count,
        "capacity_available_pct should be between 0 and 1."
    )

display(spark.createDataFrame(validation_rows).filter(F.col("check_name").contains("alternate")))

In [0]:
bad_scorecard_supplier_count = scalar(f"""
SELECT COUNT(*)
FROM {BRONZE_SCHEMA}.bronze_supplier_scorecards sc
LEFT JOIN {BRONZE_SCHEMA}.bronze_suppliers s
    ON sc.supplier_id = s.supplier_id
WHERE s.supplier_id IS NULL
""")

add_check(
    "scorecards_reference_valid_suppliers",
    "PASS" if bad_scorecard_supplier_count == 0 else "FAIL",
    bad_scorecard_supplier_count,
    "Every scorecard row should reference a valid supplier."
)

scorecard_metric_checks = [
    "fill_rate",
    "on_time_delivery_rate",
    "quality_issue_rate",
    "defect_rate"
]

for metric in scorecard_metric_checks:
    if column_exists("bronze_supplier_scorecards", metric):
        bad_metric_count = scalar(f"""
        SELECT COUNT(*)
        FROM {BRONZE_SCHEMA}.bronze_supplier_scorecards
        WHERE {metric} < 0 OR {metric} > 1
        """)

        add_check(
            f"{metric}_between_zero_and_one",
            "PASS" if bad_metric_count == 0 else "FAIL",
            bad_metric_count,
            f"{metric} should be between 0 and 1."
        )

display(spark.createDataFrame(validation_rows).filter(F.col("check_name").contains("scorecard") | F.col("check_name").contains("rate") | F.col("check_name").contains("defect")))

In [0]:
bad_po_supplier_count = scalar(f"""
SELECT COUNT(*)
FROM {BRONZE_SCHEMA}.bronze_purchase_orders po
LEFT JOIN {BRONZE_SCHEMA}.bronze_suppliers s
    ON po.supplier_id = s.supplier_id
WHERE s.supplier_id IS NULL
""")

add_check(
    "purchase_orders_reference_valid_suppliers",
    "PASS" if bad_po_supplier_count == 0 else "FAIL",
    bad_po_supplier_count,
    "Every PO should reference a valid supplier."
)

bad_po_sku_count = scalar(f"""
SELECT COUNT(*)
FROM {BRONZE_SCHEMA}.bronze_purchase_orders po
LEFT JOIN (
    SELECT DISTINCT item_id
    FROM {BRONZE_SCHEMA}.bronze_m5_sales_train_validation
) sales
    ON po.sku_id = sales.item_id
WHERE sales.item_id IS NULL
""")

add_check(
    "purchase_orders_reference_valid_m5_skus",
    "PASS" if bad_po_sku_count == 0 else "FAIL",
    bad_po_sku_count,
    "Every PO SKU should exist as an M5 item_id."
)

bad_po_quantity_count = scalar(f"""
SELECT COUNT(*)
FROM {BRONZE_SCHEMA}.bronze_purchase_orders
WHERE quantity_ordered < 0
   OR quantity_received < 0
""")

add_check(
    "purchase_order_quantities_non_negative",
    "PASS" if bad_po_quantity_count == 0 else "FAIL",
    bad_po_quantity_count,
    "PO quantities should not be negative."
)

if column_exists("bronze_purchase_orders", "status"):
    invalid_status_count = scalar(f"""
    SELECT COUNT(*)
    FROM {BRONZE_SCHEMA}.bronze_purchase_orders
    WHERE status IS NULL
    """)

    add_check(
        "purchase_order_status_not_null",
        "PASS" if invalid_status_count == 0 else "FAIL",
        invalid_status_count,
        "PO status should be populated."
    )

display(spark.createDataFrame(validation_rows).filter(F.col("check_name").contains("purchase")))

In [0]:
bad_route_supplier_count = scalar(f"""
SELECT COUNT(*)
FROM {BRONZE_SCHEMA}.bronze_shipment_routes r
LEFT JOIN {BRONZE_SCHEMA}.bronze_suppliers s
    ON r.supplier_id = s.supplier_id
WHERE s.supplier_id IS NULL
""")

add_check(
    "shipment_routes_reference_valid_suppliers",
    "PASS" if bad_route_supplier_count == 0 else "FAIL",
    bad_route_supplier_count,
    "Every shipment route should reference a valid supplier."
)

if column_exists("bronze_shipment_routes", "standard_transit_days"):
    transit_col = "standard_transit_days"
elif column_exists("bronze_shipment_routes", "std_transit_days"):
    transit_col = "std_transit_days"
else:
    transit_col = None

if transit_col:
    bad_transit_count = scalar(f"""
    SELECT COUNT(*)
    FROM {BRONZE_SCHEMA}.bronze_shipment_routes
    WHERE {transit_col} <= 0
    """)

    add_check(
        "route_transit_days_positive",
        "PASS" if bad_transit_count == 0 else "FAIL",
        bad_transit_count,
        f"{transit_col} should be positive."
    )
else:
    add_check(
        "route_transit_days_column_exists",
        "WARN",
        0,
        "No standard_transit_days or std_transit_days column found. Check route schema before Silver."
    )

display(spark.createDataFrame(validation_rows).filter(F.col("check_name").contains("route") | F.col("check_name").contains("transit")))

In [0]:
crosswalk_duplicate_count = scalar(f"""
SELECT COUNT(*)
FROM (
    SELECT canonical_sku_id, COUNT(*) AS cnt
    FROM {BRONZE_SCHEMA}.bronze_product_crosswalk
    GROUP BY canonical_sku_id
    HAVING cnt > 1
)
""")

add_check(
    "product_crosswalk_canonical_sku_id_unique",
    "PASS" if crosswalk_duplicate_count == 0 else "FAIL",
    crosswalk_duplicate_count,
    "canonical_sku_id should be unique in the Bronze product crosswalk."
)

crosswalk_null_count = scalar(f"""
SELECT COUNT(*)
FROM {BRONZE_SCHEMA}.bronze_product_crosswalk
WHERE canonical_sku_id IS NULL
   OR source_system IS NULL
   OR source_product_id IS NULL
""")

add_check(
    "product_crosswalk_required_fields_not_null",
    "PASS" if crosswalk_null_count == 0 else "FAIL",
    crosswalk_null_count,
    "canonical_sku_id, source_system, and source_product_id should be populated."
)

display(spark.createDataFrame(validation_rows).filter(F.col("check_name").contains("crosswalk")))

In [0]:
lineage_tables = [
    "bronze_suppliers",
    "bronze_supplier_aliases",
    "bronze_supplier_sku_map",
    "bronze_alternate_suppliers",
    "bronze_supplier_scorecards",
    "bronze_purchase_orders",
    "bronze_shipment_routes",
    "bronze_product_crosswalk"
]

for table_name in lineage_tables:
    if not table_exists(table_name):
        continue

    if not column_exists(table_name, "synthetic_batch_id"):
        add_check(
            f"{table_name}_synthetic_batch_id_column_exists",
            "FAIL",
            0,
            f"{table_name} is missing synthetic_batch_id."
        )
        continue

    if not column_exists(table_name, "is_synthetic"):
        add_check(
            f"{table_name}_is_synthetic_column_exists",
            "FAIL",
            0,
            f"{table_name} is missing is_synthetic."
        )
        continue

    missing_lineage_count = scalar(f"""
    SELECT COUNT(*)
    FROM {BRONZE_SCHEMA}.{table_name}
    WHERE synthetic_batch_id IS NULL
       OR is_synthetic IS NULL
       OR is_synthetic != true
    """)

    add_check(
        f"{table_name}_lineage_fields_populated",
        "PASS" if missing_lineage_count == 0 else "FAIL",
        missing_lineage_count,
        f"{table_name} should have synthetic_batch_id and is_synthetic populated."
    )

display(spark.createDataFrame(validation_rows).filter(F.col("check_name").contains("lineage") | F.col("check_name").contains("synthetic")))

In [0]:
validation_df = spark.createDataFrame(validation_rows)

(
    validation_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{BRONZE_SCHEMA}.bronze_synthetic_relationship_validation_results")
)

display(validation_df.orderBy("status", "check_name"))

In [0]:
summary_df = (
    spark.table(f"{BRONZE_SCHEMA}.bronze_synthetic_relationship_validation_results")
    .groupBy("status")
    .count()
    .orderBy("status")
)

display(summary_df)

In [0]:
failures_df = (
    spark.table(f"{BRONZE_SCHEMA}.bronze_synthetic_relationship_validation_results")
    .filter(F.col("status") == "FAIL")
    .orderBy("check_name")
)

display(failures_df)

In [0]:
%sql
SELECT
    canonical_sku_id,
    COUNT(*) AS duplicate_count,
    COLLECT_SET(source_system) AS source_systems,
    COLLECT_SET(source_product_id) AS source_product_ids,
    COLLECT_SET(source_product_name) AS product_names,
    COLLECT_SET(source_category) AS categories,
    COLLECT_SET(source_department) AS departments
FROM supplysage_bronze.bronze_product_crosswalk
GROUP BY canonical_sku_id
HAVING COUNT(*) > 1
ORDER BY duplicate_count DESC, canonical_sku_id;

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {BRONZE_SCHEMA}.bronze_product_crosswalk AS

WITH m5_products AS (
    SELECT
        CONCAT('M5_', item_id) AS canonical_sku_id,
        'm5' AS source_system,
        CAST(item_id AS STRING) AS source_product_id,
        CAST(NULL AS STRING) AS source_product_name,
        CAST(MAX(cat_id) AS STRING) AS source_category,
        CAST(MAX(dept_id) AS STRING) AS source_department,
        1.00 AS confidence,
        'native_m5_item_mapping' AS mapping_rule,
        '{SYNTHETIC_BATCH_ID}' AS synthetic_batch_id,
        true AS is_synthetic
    FROM {BRONZE_SCHEMA}.bronze_m5_sales_train_validation
    GROUP BY item_id
),

retail_products AS (
    SELECT
        CONCAT('RET_', CAST(`Product ID` AS STRING)) AS canonical_sku_id,
        'retail_inventory' AS source_system,
        CAST(`Product ID` AS STRING) AS source_product_id,
        CAST(NULL AS STRING) AS source_product_name,
        CAST(MAX(Category) AS STRING) AS source_category,
        CAST(NULL AS STRING) AS source_department,
        1.00 AS confidence,
        'native_retail_inventory_product_mapping' AS mapping_rule,
        '{SYNTHETIC_BATCH_ID}' AS synthetic_batch_id,
        true AS is_synthetic
    FROM {BRONZE_SCHEMA}.bronze_retail_inventory
    GROUP BY `Product ID`
),

dataco_products AS (
    SELECT
        CONCAT('DATACO_', CAST(`Product Card Id` AS STRING)) AS canonical_sku_id,
        'dataco' AS source_system,
        CAST(`Product Card Id` AS STRING) AS source_product_id,
        CAST(MAX(`Product Name`) AS STRING) AS source_product_name,
        CAST(MAX(`Category Name`) AS STRING) AS source_category,
        CAST(MAX(`Department Name`) AS STRING) AS source_department,
        1.00 AS confidence,
        'native_dataco_product_mapping' AS mapping_rule,
        '{SYNTHETIC_BATCH_ID}' AS synthetic_batch_id,
        true AS is_synthetic
    FROM {BRONZE_SCHEMA}.bronze_dataco_supply_chain
    WHERE `Product Card Id` IS NOT NULL
    GROUP BY `Product Card Id`
)

SELECT *
FROM m5_products

UNION ALL

SELECT *
FROM retail_products

UNION ALL

SELECT *
FROM dataco_products
""")

display(spark.table(f"{BRONZE_SCHEMA}.bronze_product_crosswalk").limit(20))

In [0]:
%sql
SELECT COUNT(*) AS duplicate_canonical_sku_count
FROM (
    SELECT canonical_sku_id, COUNT(*) AS cnt
    FROM supplysage_bronze.bronze_product_crosswalk
    GROUP BY canonical_sku_id
    HAVING cnt > 1
);

In [0]:
%sql
SELECT
    COUNT(*) AS duplicate_canonical_sku_count
FROM (
    SELECT
        canonical_sku_id,
        COUNT(*) AS cnt
    FROM supplysage_bronze.bronze_product_crosswalk
    GROUP BY canonical_sku_id
    HAVING COUNT(*) > 1
);

In [0]:
validation_rows = [
    row for row in validation_rows
    if not row["check_name"].startswith("product_crosswalk_")
]

print("Removed old product crosswalk validation rows.")
print("Remaining validation checks:", len(validation_rows))

In [0]:
lineage_tables = [
    "bronze_suppliers",
    "bronze_supplier_aliases",
    "bronze_supplier_sku_map",
    "bronze_alternate_suppliers",
    "bronze_supplier_scorecards",
    "bronze_purchase_orders",
    "bronze_shipment_routes",
    "bronze_product_crosswalk"
]

for table_name in lineage_tables:
    if not table_exists(table_name):
        continue

    if not column_exists(table_name, "synthetic_batch_id"):
        add_check(
            f"{table_name}_synthetic_batch_id_column_exists",
            "FAIL",
            0,
            f"{table_name} is missing synthetic_batch_id."
        )
        continue

    if not column_exists(table_name, "is_synthetic"):
        add_check(
            f"{table_name}_is_synthetic_column_exists",
            "FAIL",
            0,
            f"{table_name} is missing is_synthetic."
        )
        continue

    missing_lineage_count = scalar(f"""
    SELECT COUNT(*)
    FROM {BRONZE_SCHEMA}.{table_name}
    WHERE synthetic_batch_id IS NULL
       OR is_synthetic IS NULL
       OR is_synthetic != true
    """)

    add_check(
        f"{table_name}_lineage_fields_populated",
        "PASS" if missing_lineage_count == 0 else "FAIL",
        missing_lineage_count,
        f"{table_name} should have synthetic_batch_id and is_synthetic populated."
    )

display(
    spark.createDataFrame(validation_rows)
    .filter(
        F.col("check_name").contains("lineage") |
        F.col("check_name").contains("synthetic_batch_id") |
        F.col("check_name").contains("is_synthetic")
    )
)

In [0]:
validation_df = spark.createDataFrame(validation_rows)

(
    validation_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{BRONZE_SCHEMA}.bronze_synthetic_relationship_validation_results")
)

display(validation_df.orderBy("status", "check_name"))

In [0]:
summary_df = (
    spark.table(f"{BRONZE_SCHEMA}.bronze_synthetic_relationship_validation_results")
    .groupBy("status")
    .count()
    .orderBy("status")
)

display(summary_df)

In [0]:
%sql
SELECT 'bronze_suppliers' AS table_name, COUNT(*) AS row_count
FROM supplysage_bronze.bronze_suppliers

UNION ALL

SELECT 'bronze_supplier_aliases', COUNT(*)
FROM supplysage_bronze.bronze_supplier_aliases

UNION ALL

SELECT 'bronze_supplier_sku_map', COUNT(*)
FROM supplysage_bronze.bronze_supplier_sku_map

UNION ALL

SELECT 'bronze_alternate_suppliers', COUNT(*)
FROM supplysage_bronze.bronze_alternate_suppliers

UNION ALL

SELECT 'bronze_supplier_scorecards', COUNT(*)
FROM supplysage_bronze.bronze_supplier_scorecards

UNION ALL

SELECT 'bronze_purchase_orders', COUNT(*)
FROM supplysage_bronze.bronze_purchase_orders

UNION ALL

SELECT 'bronze_shipment_routes', COUNT(*)
FROM supplysage_bronze.bronze_shipment_routes

UNION ALL

SELECT 'bronze_product_crosswalk', COUNT(*)
FROM supplysage_bronze.bronze_product_crosswalk

UNION ALL

SELECT 'bronze_synthetic_generation_manifest', COUNT(*)
FROM supplysage_bronze.bronze_synthetic_generation_manifest

UNION ALL

SELECT 'bronze_synthetic_relationship_validation_results', COUNT(*)
FROM supplysage_bronze.bronze_synthetic_relationship_validation_results;

In [0]:
fail_count = scalar(f"""
SELECT COUNT(*)
FROM {BRONZE_SCHEMA}.bronze_synthetic_relationship_validation_results
WHERE status = 'FAIL'
""")

warn_count = scalar(f"""
SELECT COUNT(*)
FROM {BRONZE_SCHEMA}.bronze_synthetic_relationship_validation_results
WHERE status = 'WARN'
""")

if fail_count == 0:
    print("✅ Notebook 04 PASSED: Synthetic Bronze lineage and relationship validation is complete.")
    print(f"WARN count: {warn_count}")
    print("Next notebook: 05_bronze_ingest_external_risk_sources_v0")
else:
    print("❌ Notebook 04 has failures. Do not move forward yet.")
    print(f"FAIL count: {fail_count}")
    print("Inspect failed checks before continuing.")